Install Dependencies

In [ ]:
!pip install -q bitsandbytes accelerate peft transformers datasets trl fsspec==2025.3.2 -U

Convert Dataset for Training

In [ ]:
import json

input_file = "training_data_multiturn.jsonl"
output_file = "training_data_for_lora.jsonl"

with open(input_file, "r", encoding="utf-8") as f_in, open(output_file, "w", encoding="utf-8") as f_out:
    for line in f_in:
        data = json.loads(line)
        converted = {
            "instruction": data["prompt"].strip(),
            "output": data["response"].strip()
        }
        json.dump(converted, f_out, ensure_ascii=False)
        f_out.write("\\n")

print(f"✅ Converted to {output_file}")


Load Converted Dataset

In [ ]:
from datasets import load_dataset

# Load dataset in instruction/output format
dataset = load_dataset("json", data_files="training_data_for_lora.jsonl")

# Optional: create train/test split (5% for eval)
dataset = dataset["train"].train_test_split(test_size=0.05)

# Preview the structure
dataset


DatasetDict({
    train: Dataset({
        features: ['full_prompt'],
        num_rows: 31968
    })
    test: Dataset({
        features: ['full_prompt'],
        num_rows: 3553
    })
})

Log in to Hugging Face

In [ ]:
from huggingface_hub import login

login()


Load Model + Tokenizer (4-bit Quantized)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# Load base model + tokenizer
model_name = "mistralai/Mistral-7B-v0.1"

# Use bfloat16 if supported, else fallback to float16
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = dict(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", **bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Prepare for LoRA fine-tuning
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("✅ Model loaded, quantized, and LoRA-ready.")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Train Model with LoRA

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

steps_per_epoch = len(dataset["train"]) // (1 * 4)

training_args = TrainingArguments(
    output_dir="gurt-mistral-lora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    eval_steps=steps_per_epoch,
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="instruction",  # <-- updated field
    args=training_args
)

trainer.train()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:292: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/31968 [00:00<?, ? examples/s]

Map:   0%|          | 0/3553 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:396: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,3.549000
20,3.196000
30,2.997400
40,2.973500
50,2.993300
60,2.839200
70,2.919300
80,2.794700
90,2.871800
100,2.869300


Step,Training Loss
10,3.549000
20,3.196000
30,2.997400
40,2.973500
50,2.993300
60,2.839200
70,2.919300
80,2.794700
90,2.871800
100,2.869300


TrainOutput(global_step=15984, training_loss=2.2513892233550727, metrics={'train_runtime': 15537.8428, 'train_samples_per_second': 4.115, 'train_steps_per_second': 1.029, 'total_flos': 1.4898032811196416e+17, 'train_loss': 2.2513892233550727, 'epoch': 2.0})

Save LoRA Adapter & Tokenizer + Export to Drive

In [ ]:
# Define where to save in Colab
save_dir = "/content/gurt-mistral-lora"

# Save LoRA adapter and tokenizer
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"✅ Model and tokenizer saved to: {save_dir}")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy to Google Drive
!cp -r {save_dir} /content/drive/MyDrive/
print("✅ Backup saved to Google Drive at: /MyDrive/gurt-mistral-lora")


Model and tokenizer saved to: /content/mistral-whatsapp-final
Mounted at /content/drive


(If Necessary) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(If Necessary) Load Model from Drive

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Load from Drive
model_path = "/content/drive/MyDrive/mistral-whatsapp-final"

# 4-bit quantization setup
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, model_path)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Generating a Response

In [ ]:
import torch

def chat_response(prompt, max_new_tokens=120):
    # Append bot name if needed
    prompt += "\nGurt:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract Gurt's reply
    if "Gurt:" in decoded:
        reply = decoded.split("Gurt:")[-1].strip().split("\n")[0]
    else:
        reply = decoded[len(prompt):].strip().split("\n")[0]

    return reply


Testing

In [ ]:
prompt = """Petr: alain or casper?"""

response = chat_response(prompt)
print("Generated Response:", response)


Generated Response: Yeah
